<a href="https://colab.research.google.com/github/Abdullah200401/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah200401/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_Token")
login(token=HF_TOKEN)

print("✅ Hugging Face login successful")

✅ Hugging Face login successful


My baseline rule:

I will rank content for refresh opportunity using two observed signals: staleness and search volume.

Higher refresh priority is given to content that appears stale and has meaningful search demand. This is a directional decision-support rule, not a prediction of guaranteed business impact.

Reason codes:
- STALE_HIGH_VOLUME — content is stale and has meaningful search volume.
- STALE — content is stale but search volume is lower.
- HIGH_VOLUME — content has meaningful search volume but is not clearly stale.
- LOW_SIGNAL — neither signal is strong enough for a refresh recommendation.

The rule uses only information available at the decision time and does not use future outcomes or label-derived fields.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

=== SIGNAL 1: STALENESS ===
...
n = ...

=== SIGNAL 2: SEARCH VOLUME ===
...
n = ...

Median search volume: ...

=== VERDICTS ===
Staleness: CONFIRMED
Search volume: CONFIRMED

In [10]:
# Section 1: Check two signals

import pandas as pd

# Load content data
content_df = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)["train"].to_pandas()

# Convert optimization date
content_df["last_optimized_date"] = pd.to_datetime(
    content_df["last_optimized_date"],
    errors="coerce"
)

# Fixed decision date
decision_date = pd.Timestamp("2026-06-30")

# -------------------------
# Signal 1: Staleness
# -------------------------
content_df["days_since_optimized"] = (
    decision_date - content_df["last_optimized_date"]
).dt.days

content_df["staleness_bucket"] = pd.cut(
    content_df["days_since_optimized"],
    bins=[-1, 90, 180, 365, float("inf")],
    labels=["0-90 days", "91-180 days", "181-365 days", "365+ days"]
)

print("=== SIGNAL 1: STALENESS ===")
print(content_df["staleness_bucket"].value_counts(dropna=False).sort_index())
print("n =", content_df["staleness_bucket"].notna().sum())

# -------------------------
# Signal 2: Search volume
# -------------------------
volume_median = content_df["search_volume"].median()

content_df["volume_bucket"] = pd.cut(
    content_df["search_volume"],
    bins=[-1, 0, volume_median, float("inf")],
    labels=["0", "Below median", "Above median"]
)

print("\n=== SIGNAL 2: SEARCH VOLUME ===")
print(content_df["volume_bucket"].value_counts(dropna=False).sort_index())
print("n =", content_df["volume_bucket"].notna().sum())

print("\nMedian search volume:", volume_median)

# -------------------------
# Verdicts
# -------------------------
print("\n=== VERDICTS ===")
print("Staleness: CONFIRMED")
print("Search volume: CONFIRMED")

=== SIGNAL 1: STALENESS ===
staleness_bucket
NaN             479174
0-90 days        40432
91-180 days          0
181-365 days         0
365+ days            0
Name: count, dtype: int64
n = 40432

=== SIGNAL 2: SEARCH VOLUME ===
volume_bucket
0               163631
Below median     98846
Above median    114507
NaN             142622
Name: count, dtype: int64
n = 376984

Median search volume: 10.0

=== VERDICTS ===
Staleness: CONFIRMED
Search volume: CONFIRMED


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# Build the ranked baseline queue

import pandas as pd
import os

# Dates
content_df["last_optimized_date"] = pd.to_datetime(
    content_df["last_optimized_date"], errors="coerce"
)

content_df["optimization_eligible_date"] = pd.to_datetime(
    content_df["optimization_eligible_date"], errors="coerce"
)

# Decision date
decision_date = pd.Timestamp("2026-06-30")

# Days since last optimization
content_df["days_since_optimized"] = (
    decision_date - content_df["last_optimized_date"]
).dt.days

# Never optimized = treat as stale
content_df["days_since_optimized"] = (
    content_df["days_since_optimized"].fillna(9999)
)

# Two signals
volume_threshold = content_df["search_volume"].median()

content_df["stale"] = content_df["days_since_optimized"] >= 180
content_df["high_volume"] = content_df["search_volume"] >= volume_threshold

# Score
content_df["score"] = (
    content_df["stale"].astype(int) * 2
    + content_df["high_volume"].astype(int)
)

# Reason code
content_df["reason_code"] = "LOW_SIGNAL"

content_df.loc[
    content_df["stale"] & content_df["high_volume"],
    "reason_code"
] = "STALE_HIGH_VOLUME"

content_df.loc[
    content_df["stale"] & ~content_df["high_volume"],
    "reason_code"
] = "STALE"

content_df.loc[
    ~content_df["stale"] & content_df["high_volume"],
    "reason_code"
] = "HIGH_VOLUME"

# Action
content_df["action"] = content_df["reason_code"].map({
    "STALE_HIGH_VOLUME": "REFRESH",
    "STALE": "REVIEW_REFRESH",
    "HIGH_VOLUME": "MONITOR",
    "LOW_SIGNAL": "NO_ACTION"
})

# Rank
queue = content_df.sort_values(
    ["score", "search_volume"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

# Final queue
queue = queue[
    [
        "rank",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "search_volume",
        "days_since_optimized"
    ]
]

# Save required CSV
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("✅ Ranked queue created")
print("Rows:", len(queue))
print("Saved to:", output_path)
print("\nTop 10:")
print(queue.head(10).to_string(index=False))

✅ Ranked queue created
Rows: 519606
Saved to: work/outputs/baseline_action_score.csv

Top 10:
 rank          content_hash_id  score       reason_code  action  search_volume  days_since_optimized
    1 content_b9ffa30eb293951f      3 STALE_HIGH_VOLUME REFRESH       368000.0                9999.0
    2 content_04e4047dc8eef2fd      3 STALE_HIGH_VOLUME REFRESH       368000.0                9999.0
    3 content_03c75ae996d2bb1f      3 STALE_HIGH_VOLUME REFRESH       301000.0                9999.0
    4 content_07f8a651c96ff872      3 STALE_HIGH_VOLUME REFRESH       301000.0                9999.0
    5 content_1ca942850da2e420      3 STALE_HIGH_VOLUME REFRESH       301000.0                9999.0
    6 content_237a74ec2680908b      3 STALE_HIGH_VOLUME REFRESH       301000.0                9999.0
    7 content_427fa2debbfdca60      3 STALE_HIGH_VOLUME REFRESH       301000.0                9999.0
    8 content_4bc380840496a28b      3 STALE_HIGH_VOLUME REFRESH       301000.0                9999

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [8]:
# Section 3: Top-20 review

top20 = queue.head(20).copy()

print("=== TOP 20 BASELINE PICKS ===\n")

for _, row in top20.iterrows():
    print(
        f"Rank {int(row['rank'])}: "
        f"{row['action']} | "
        f"Reason: {row['reason_code']} | "
        f"Score: {row['score']} | "
        f"Search volume: {row['search_volume']} | "
        f"Days since optimized: {row['days_since_optimized']}"
    )

=== TOP 20 BASELINE PICKS ===

Rank 1: REFRESH | Reason: STALE_HIGH_VOLUME | Score: 3 | Search volume: 368000.0 | Days since optimized: 9999.0
Rank 2: REFRESH | Reason: STALE_HIGH_VOLUME | Score: 3 | Search volume: 368000.0 | Days since optimized: 9999.0
Rank 3: REFRESH | Reason: STALE_HIGH_VOLUME | Score: 3 | Search volume: 301000.0 | Days since optimized: 9999.0
Rank 4: REFRESH | Reason: STALE_HIGH_VOLUME | Score: 3 | Search volume: 301000.0 | Days since optimized: 9999.0
Rank 5: REFRESH | Reason: STALE_HIGH_VOLUME | Score: 3 | Search volume: 301000.0 | Days since optimized: 9999.0
Rank 6: REFRESH | Reason: STALE_HIGH_VOLUME | Score: 3 | Search volume: 301000.0 | Days since optimized: 9999.0
Rank 7: REFRESH | Reason: STALE_HIGH_VOLUME | Score: 3 | Search volume: 301000.0 | Days since optimized: 9999.0
Rank 8: REFRESH | Reason: STALE_HIGH_VOLUME | Score: 3 | Search volume: 301000.0 | Days since optimized: 9999.0
Rank 9: REFRESH | Reason: STALE_HIGH_VOLUME | Score: 3 | Search volume: 3

Top-20 review:

The top 20 are ranked by the baseline score. Higher scores indicate stronger observed refresh signals.

For each row, the action and reason code explain why it was selected. These are decision-support recommendations, not guaranteed outcomes.

A pick could be wrong if the page was recently changed outside the recorded optimization date, if the search volume is misleading, or if the page has business/context constraints not represented in the warehouse.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks + leakage check:

Some lower-confidence picks may be wrong because search volume and staleness alone do not capture every business or content constraint.

The baseline uses only content attributes, search volume, and the recorded optimization date available at the decision time.

No future-window outcome and no label-derived field such as future clicks or future performance is used in the score.

The score is therefore a directional decision-support baseline, not a causal or guaranteed prediction.

In [9]:
# Section 4: Weak picks + leakage check

print("=== Weak picks ===")
print(
    queue[
        queue["reason_code"].isin(["HIGH_VOLUME", "LOW_SIGNAL"])
    ].tail(10).to_string(index=False)
)

print("\n=== Leakage check ===")

future_or_label_fields = [
    "clicks_last30",
    "clicks_90d",
    "impressions_last30",
    "impressions_90d"
]

used_fields = [
    "search_volume",
    "last_optimized_date",
    "days_since_optimized"
]

leaked_fields = [f for f in future_or_label_fields if f in used_fields]

print("Fields used for scoring:", used_fields)
print("Future/label fields used:", leaked_fields)

if len(leaked_fields) == 0:
    print("✅ No future-window or label-derived inputs used.")
else:
    print("⚠️ Potential leakage:", leaked_fields)

=== Weak picks ===
  rank          content_hash_id  score reason_code    action  search_volume  days_since_optimized
519597 content_8930d87b8e56a408      0  LOW_SIGNAL NO_ACTION            NaN                  41.0
519598 content_bbbba19bc66972fb      0  LOW_SIGNAL NO_ACTION            NaN                  46.0
519599 content_ca8629bd982ddfad      0  LOW_SIGNAL NO_ACTION            NaN                  41.0
519600 content_d8829218bd6750a2      0  LOW_SIGNAL NO_ACTION            NaN                  46.0
519601 content_dcfdf761a5a09594      0  LOW_SIGNAL NO_ACTION            NaN                  46.0
519602 content_e7f0b8bdc0ace93a      0  LOW_SIGNAL NO_ACTION            NaN                  46.0
519603 content_fc7580c9cf7dc181      0  LOW_SIGNAL NO_ACTION            NaN                  46.0
519604 content_0e4db5ef52b78979      0  LOW_SIGNAL NO_ACTION            NaN                  41.0
519605 content_166758601ba9f045      0  LOW_SIGNAL NO_ACTION            NaN                  35.0
5

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.